# OctWave3 — Kaggle Image Classification (Colab T4)

**Runtime → Change runtime type → T4 GPU** before running anything.

This notebook is deliberately thin: all real logic lives in `src/*.py` in the GitHub repo.
Edit those files locally, `git push`, then re-run the **Pull repo** cell here. You should
almost never need to edit this notebook, which keeps `.ipynb` diffs tiny.

Checkpoints are written to **Google Drive**, so a disconnect costs one epoch, not the run.

## 1. Check GPU

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 2. Install extra packages

Torch/torchvision already ship with Colab — only these are missing.

In [ ]:
!pip install -q timm albumentations kaggle

## 3. Mount Google Drive

Checkpoints live here so training survives a dropped session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_OUT = Path('/content/drive/MyDrive/octwave3/outputs')
(DRIVE_OUT / 'checkpoints').mkdir(parents=True, exist_ok=True)
(DRIVE_OUT / 'submissions').mkdir(parents=True, exist_ok=True)
print(DRIVE_OUT)

## 4. Clone / pull the repo

Set `REPO_URL` once. Re-run this cell every session, and again after every push
from your laptop — it is the only step needed to get new code into Colab.

For a private repo, create a GitHub fine-grained token and use:
`https://<TOKEN>@github.com/sasindu345/OctWave3.git` (store the token in Colab Secrets 🔑, not here).

In [ ]:
import os, sys, subprocess

REPO_DIR = '/content/OctWave3'
os.chdir('/content')          # never operate from inside a dir we may delete

# A private repo needs a token. Store it in Colab Secrets (key icon) as GH_TOKEN:
# GitHub > Settings > Developer settings > Fine-grained tokens,
# repo = OctWave3, permission Contents: Read and write.
try:
    from google.colab import userdata
    TOKEN = userdata.get('GH_TOKEN')
except Exception:
    TOKEN = None

URL = (f'https://{TOKEN}@github.com/sasindu345/OctWave3.git' if TOKEN
       else 'https://github.com/sasindu345/OctWave3.git')

def run(*args, cwd=None):
    r = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    msg = (r.stdout + r.stderr).strip()
    if TOKEN:                       # a failing git echoes the URL - redact it
        msg = msg.replace(TOKEN, '***')
    if msg:
        print(msg)
    return r.returncode

if os.path.isdir(REPO_DIR + '/.git'):
    run('git', 'pull', '-q', URL, 'main', cwd=REPO_DIR)
else:
    if run('git', 'clone', '-q', URL, REPO_DIR) != 0:
        raise SystemExit(
            'clone failed.\n'
            '  - private repo? create the GH_TOKEN secret (key icon, left sidebar)\n'
            '    and make sure "Notebook access" is toggled ON\n'
            '  - public repo? check the URL is right')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
run('git', 'log', '--oneline', '-3')

## 5. Download the competition data

Upload your `kaggle.json` (Kaggle → Account → Create New API Token) when prompted,
or store its contents in Colab Secrets. Data goes to local disk (`/content`), **not**
Drive — local disk is far faster to read from, and re-downloading each session is cheap.

In [ ]:
from google.colab import files
import os

if not os.path.exists('/root/.kaggle/kaggle.json'):
    files.upload()                      # pick kaggle.json
    !mkdir -p /root/.kaggle && mv kaggle.json /root/.kaggle/
    !chmod 600 /root/.kaggle/kaggle.json

COMP = 'oct-wave-3-0-kaggle-challenge-02'
!kaggle competitions download -c $COMP -p /content/data
!unzip -q -o '/content/data/*.zip' -d /content/data
# images.zip sits INSIDE the competition zip - unzip that too
!if [ -f /content/data/images.zip ]; then unzip -q -o /content/data/images.zip -d /content/data; fi
!ls /content/data && echo "--- images:" && ls /content/data/images | head -3
# the outer zip is 456MB of dead weight once unzipped
!rm -f /content/data/oct-wave-3-0-kaggle-challenge-02.zip


## 5b. Inspect the data — run this BEFORE configuring anything

Do not assume the layout, the class count, or the submission format. Print them.
Every number this cell reports is a fact; anything not printed here is a guess.

Paste the whole output when asking for help — it answers most setup questions at once.

In [ ]:
import pandas as pd, numpy as np, cv2
from pathlib import Path

D = Path('/content/data')

print('=' * 60, '\n1. TOP-LEVEL FILES\n', '=' * 60)
for p in sorted(D.iterdir()):
    kind = 'dir ' if p.is_dir() else 'file'
    size = '' if p.is_dir() else f'{p.stat().st_size/1e6:.1f} MB'
    print(f'  [{kind}] {p.name:40s} {size}')

print('\n', '=' * 60, '\n2. CSV FILES\n', '=' * 60)
for csv in sorted(D.rglob('*.csv')):
    df = pd.read_csv(csv)
    print(f'\n--- {csv.relative_to(D)}  shape={df.shape}')
    print('    columns:', list(df.columns))
    print(df.head(3).to_string(max_colwidth=30))

print('\n', '=' * 60, '\n3. IMAGE FOLDERS\n', '=' * 60)
for d in sorted(x for x in D.rglob('*') if x.is_dir()):
    imgs = [f for f in d.iterdir() if f.suffix.lower() in
            {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}]
    subs = [x for x in d.iterdir() if x.is_dir()]
    if imgs or subs:
        print(f'  {str(d.relative_to(D)):45s} {len(imgs):>7} images, {len(subs):>4} subfolders')

print('\n', '=' * 60, '\n4. SAMPLE IMAGES\n', '=' * 60)
all_imgs = [f for f in D.rglob('*') if f.suffix.lower() in {'.jpg', '.jpeg', '.png'}]
print(f'  total image files: {len(all_imgs)}')
for f in all_imgs[:5]:
    im = cv2.imread(str(f))
    print(f'  {f.name:35s} shape={None if im is None else im.shape}')
shapes = {}
for f in np.random.default_rng(0).choice(all_imgs, min(200, len(all_imgs)), replace=False):
    im = cv2.imread(str(f))
    if im is not None:
        shapes[im.shape] = shapes.get(im.shape, 0) + 1
print('  shape distribution (200 sampled):', dict(sorted(shapes.items(), key=lambda x: -x[1])[:5]))

print('\n', '=' * 60, '\n5. CLASS BALANCE (from train.csv)\n', '=' * 60)
train_csv = D / 'train.csv'
if train_csv.exists():
    tr = pd.read_csv(train_csv)
    vc = tr['appearance'].value_counts().sort_index()
    names = {0: 'neither', 1: 'Tom only', 2: 'Jerry only', 3: 'both'}
    for k, v in vc.items():
        bar = '#' * int(60 * v / vc.max())
        print(f'  {k} {names.get(k, "?"):12s} {v:6d}  ({100*v/len(tr):5.1f}%)  {bar}')
    print(f'\n  total train: {len(tr)}   imbalance ratio (max/min): {vc.max()/vc.min():.1f}x')
    print(f'  rarest class has {vc.min()} images -> ~{vc.min()//5} per validation fold')
else:
    print('  train.csv not found')

print('\n', '=' * 60, '\n6. DISK\n', '=' * 60)
!df -h /content | tail -1


## 6. Configure the experiment

Everything tunable is in `src/config.py`; override per-run here.

In [ ]:
import sys, os, importlib
REPO_DIR = '/content/OctWave3'
os.chdir(REPO_DIR)                       # survives a runtime restart
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
# a path added while the dir was absent gets cached as "empty" - clear that
sys.path_importer_cache.pop(REPO_DIR, None)
importlib.invalidate_caches()

from pathlib import Path
import importlib, src.config, src.dataset, src.model, src.train, src.utils
for m in (src.config, src.utils, src.dataset, src.model, src.train):
    importlib.reload(m)
from src.config import cfg

cfg.data_dir = Path('/content/data')
cfg.out_dir  = DRIVE_OUT          # checkpoints survive disconnects

cfg.exp_name    = 'exp01_baseline'
cfg.model_name  = 'tf_efficientnet_b0'
cfg.img_size    = 224
cfg.batch_size  = 32
cfg.epochs      = 12
cfg.lr          = 3e-4

# Fixed by the competition - do not change these:
#   4 classes (0 neither / 1 Tom / 2 Jerry / 3 both), metric = macro F1
cfg.num_classes   = 4
cfg.metric        = 'macro_f1'
cfg.class_weights = True          # severe imbalance + macro F1
cfg.label_smoothing = 0.05        # train labels are noisy, test labels are clean

cfg.to_dict()

## 7. Train

Safe to re-run after a disconnect — it resumes from `*_last.pt` in Drive automatically.
Keep the browser tab open; Colab kills idle sessions.

In [ ]:
import sys, os, importlib
REPO_DIR = '/content/OctWave3'
os.chdir(REPO_DIR)                       # survives a runtime restart
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
# a path added while the dir was absent gets cached as "empty" - clear that
sys.path_importer_cache.pop(REPO_DIR, None)
importlib.invalidate_caches()

from src.train import run_fold
best = run_fold(cfg, fold=0)

## 8. Training curves

Reads the `run_log.jsonl` that `train.py` appends to on every epoch.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

log = pd.read_json(cfg.out_dir / 'run_log.jsonl', lines=True)
log = log[log.exp == cfg.exp_name]

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(log.epoch, log.train_loss, label='train')
ax[0].plot(log.epoch, log.val_loss, label='valid')
ax[0].set_title('loss'); ax[0].set_xlabel('epoch'); ax[0].legend()
ax[1].plot(log.epoch, log.acc, label='acc')
ax[1].plot(log.epoch, log.f1, label='macro f1')
ax[1].set_title('metrics'); ax[1].set_xlabel('epoch'); ax[1].legend()
plt.tight_layout(); plt.show()
log.tail()

## 9. Predict & submit

In [ ]:
import sys, os, importlib
REPO_DIR = '/content/OctWave3'
os.chdir(REPO_DIR)                       # survives a runtime restart
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
# a path added while the dir was absent gets cached as "empty" - clear that
sys.path_importer_cache.pop(REPO_DIR, None)
importlib.invalidate_caches()

from src.dataset import build_test_dataframe
from src.predict import predict, make_submission

test_df = build_test_dataframe(cfg)
print(f'{len(test_df)} test images')

ckpts = sorted((cfg.out_dir / 'checkpoints').glob(f'{cfg.exp_name}_f*_best.pt'))
print('using checkpoints:', [c.name for c in ckpts])

probs = predict(cfg, ckpts, test_df)
sub   = make_submission(cfg, probs, test_df, f'{cfg.exp_name}.csv')
sub.head()

In [ ]:
!kaggle competitions submit -c $COMP \
    -f {cfg.out_dir}/submissions/{cfg.exp_name}.csv \
    -m "{cfg.exp_name}: {cfg.model_name} @ {cfg.img_size}"

## 10. Send the results back

Colab is a separate machine — nothing leaves it automatically. Pick one:

**A. Copy-paste (no setup).** Select a cell's output, copy, paste it where you need it.
Enough for almost everything: errors, epoch logs, and the `decide()` verdict are all
just a few lines of text.

**B. Push to GitHub (below).** Sends the run log and figures back to the repo so they
can be read off-Colab. Only small files travel — checkpoints and raw predictions stay
in Drive.

One-time setup for B: GitHub → Settings → Developer settings → Fine-grained tokens →
new token, this repo, **Contents: Read and write**. Then in Colab click the 🔑 key icon
in the left sidebar and add a secret named `GH_TOKEN` with that value.

In [ ]:
import shutil, subprocess
from pathlib import Path
from google.colab import userdata

REPO    = Path('/content/OctWave3')
RESULTS = REPO / 'results'
(RESULTS / 'figures').mkdir(parents=True, exist_ok=True)

# Only the small evidence files travel. Checkpoints (~50MB) and raw
# probabilities stay in Drive - git is the wrong place for them.
log = DRIVE_OUT / 'run_log.jsonl'
if log.exists():
    shutil.copy(log, RESULTS / 'run_log.jsonl')
for png in (DRIVE_OUT / 'figures').glob('*.png'):
    shutil.copy(png, RESULTS / 'figures' / png.name)
print('staged:', [p.name for p in RESULTS.rglob('*') if p.is_file()])

TOKEN = userdata.get('GH_TOKEN')

def git(*args, secret=False):
    r = subprocess.run(('git',) + args, cwd=REPO, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if secret:                                   # never print a URL holding the token
        out = out.replace(TOKEN, '***') if TOKEN else out
    if out:
        print(out)
    return r.returncode

git('config', 'user.email', 'cryptxgmora@gmail.com')
git('config', 'user.name', 'sasindu345')
git('add', 'results', 'logs')
if git('diff', '--cached', '--quiet') == 0:
    print('nothing new to push')
else:
    git('commit', '-m', f'results: {cfg.exp_name}')
    git('push', f'https://{TOKEN}@github.com/sasindu345/OctWave3.git', 'HEAD:main', secret=True)
    print('\npushed - now run `git pull` on your laptop')

## 11. Record the run

1. One row in `logs/EXPERIMENTS.md` — model, image size, LR, CV, LB.
2. One entry in `logs/DECISIONS.md` if this run decided something (use the
   `decide()` output from notebook 02, not your impression of the numbers).
3. Anything you changed in `src/` goes in `logs/CHANGELOG.md`.

Then open **`02_analysis.ipynb`** — that is where you find out whether the change
was real or just noise.

If you edited *this notebook*: `Edit → Clear all outputs` first, then
`File → Save a copy in GitHub` → path `notebooks/01_train_colab.ipynb`.
Clearing outputs matters — it keeps the token and the image blobs out of the commit.